# DNA nanorulers (GATTA-PAINT 80Y and 80R) — localisation accuracy

**Makes up Figure(s):** S76 (80Y, Cy3B imager) and S77 (80R, ATTO 655 imager), panels a) and b):
`Nanoruler_80Y_render.svg`, `Nanoruler_80R_render.svg` (section 5); panels c) to f):
`Nanoruler_80Y_distances.svg`, `Nanoruler_80R_distances.svg` (section 6).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/Revision_2/20260910_DNA_Nanorulers/`.
GATTAquant GATTA-PAINT 80R / 80Y nanorulers: three docking sites on a line, 80 nm apart
(± 5 nm, as written on the slides), imaged by DNA-PAINT on the Ximea camera.

**What changed from the published version:** the published renders came from HDBSCAN-clustered
localisations, with an extra fiducial-removal step. That was needed while the fitter had the x/y
ROI-origin bug (fixed 2026-10-28), which smeared every position by up to one camera pixel. With the
fixed fit, the panels are rendered straight from the drift-corrected, linked localisations.

## FAST_MODE

Each acquisition is 9 (80R) or 11 (80Y) TIFFs of ~4.3 GB, plus a ~2.2 GB metadata file.
`FAST_MODE = True` (default) downloads and fits only the first TIFF of each (~6,500 frames), which
runs the whole pipeline but samples each ruler far less than the published numbers.
`FAST_MODE = False` downloads and fits everything (**~85 GB, hours of fitting**; run in
`tmux`/`screen`). An existing `Localisations.h5` made with the same settings is reused.

---
## 0 — Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py and _nanorulers.py live one level up (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _nanorulers as nru

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)
print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF of each acquisition (illustrative). False: everything (~85 GB).
REFIT = False      # True: fit again even if a matching Localisations.h5 exists

PIXEL_NM = 69.0    # Ximea, sample plane
DESIGN_NM = 80.0   # site spacing (GATTA-PAINT 80R / 80Y)
TOLERANCE_NM = 5.0 # as written on the slides

REMOTE_ROOT = "2026_Multicolour_Paper/Ximea/Revision_2/20260910_DNA_Nanorulers/"
# Per slide: deposition folder (below REMOTE_ROOT), detection wavelength (um), and the cuts of
# the original analysis (precision/PSF-width in nm, minimum colour fractions).
RULERS = {
    "80Y": dict(folder="80Y_NanoRuler/20p561_DC1_LP561_BP582_64_75ms_TIRF_1",
                dye="Cy3B", peak_wavelength=0.55,
                cuts=dict(max_err_nm=40.0, sigma_nm=(80.0, 175.0),
                          min_colour={"A_B": 0.005, "A_G": 0.05, "A_R": 0.4}),
                aim_segmentation=10),
    "80R": dict(folder="80R_NanoRuler/60mW_638_1_DC1_LP635_75ms_TIRF_1",
                dye="ATTO 655", peak_wavelength=0.65,
                cuts=dict(max_err_nm=40.0, sigma_nm=(100.0, 200.0),
                          min_colour={"A_B": 0.005, "A_G": 0.05, "A_R": 0.5}),
                aim_segmentation=20),
}
FIT_PARAMS = dict(pfa=1e-3, ROI_size=12, NA=1.49, pixel_size=PIXEL_NM / 1000, image_type=".tif")
LINK = dict(intersect_nm=20.0, roi_nm=60.0, max_dark_time=2)   # AIM and linking (nm, frames)

SupRes_F = SR_Functions.SuperRes_Functions(camera="ximea", pixel_size=PIXEL_NM / 1000)
import types
smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5}, extent=1.5, data_arg="image",
    smoothing_function=sCMOS.gaussian_filter_stack)

# ── Paths ──────────────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().resolve().parents[2]
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Nanorulers"   # git-ignored (bare outputs/ rule)
DATA_DIR  = OUT_ROOT / "data"                # downloads; one folder per acquisition
FIT_ROOT  = DATA_DIR                         # where each acquisition's Localisations.h5 is
FIG_DIR   = OUT_ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

calib = {k: IO.read_tiff(CALIB_DIR / f"{f}.tif") for k, f in
         [("gain_map", "gain"), ("offset_map", "offset"), ("variance", "variance"),
          ("read_noise", "readnoise"), ("rqe", "rqe")]}

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Fits       : {FIT_ROOT.resolve()}")
print(f"Outputs    : {OUT_ROOT.resolve()}")

---
## 2 — Download and fit (skipped when a matching fit exists)

Downloads the acquisition (`FAST_MODE`: its first TIFF) plus its metadata, then fits every frame
with `SuperRes_Functions.fit_imaging_data`, writing `Localisations.h5` beside the TIFFs. Nothing is
downloaded when a fit with the same settings is already there.

In [ ]:
fits = {}
for name, r in RULERS.items():
    folder = FIT_ROOT / r["folder"]
    record = {"fast_mode": FAST_MODE, "peak_wavelength": r["peak_wavelength"], **FIT_PARAMS}

    def fit(folder=folder, r=r):
        bd.download_acquisition(REMOTE_ROOT + r["folder"], folder, first_file_only=FAST_MODE)
        SupRes_F.fit_imaging_data(
            str(folder), smoothing_function, calib["gain_map"], calib["offset_map"],
            calib["rqe"], calib["read_noise"], variance=calib["variance"],
            peak_wavelength=r["peak_wavelength"], **FIT_PARAMS)

    fits[name] = nru.fit_or_reuse(folder, record, fit, refit=REFIT)

---
## 3 — Quality cuts, drift correction, linking

The cuts of the original analysis (section 1), then AIM drift correction and linking of repeat
detections of one binding event (within the median precision, up to 2 dark frames). The linked
table is cached in `OUT_ROOT` with its settings.

In [ ]:
linked, acq = {}, {}
for name, r in RULERS.items():
    width, height, n_frames = nru.acquisition_info(fits[name].parent, IO)
    acq[name] = dict(width=width, height=height, n_frames=n_frames)
    locs = pd.read_hdf(fits[name])
    kept, cut_table = nru.quality_cuts(locs, PIXEL_NM, **r["cuts"])
    print(f"{name}: {len(locs):,} localisations, {width} x {height} px, {n_frames:,} frames")
    print(cut_table.to_string(index=False))
    del locs
    settings = {**LINK, "segmentation": r["aim_segmentation"], "cuts": r["cuts"]}
    linked[name], drift = nru.undrift_and_link(
        kept, width, height, n_frames, PIXEL_NM, OUT_ROOT / f"{name}_linked.h5", settings)
    linked[name] = nru.slim(linked[name])   # only the columns used below
    del kept
    print(f"{name}: {len(linked[name]):,} linked events, median precision "
          f"{linked[name][['xc_err', 'yc_err']].median().mean() * PIXEL_NM:.1f} nm\n")
    if drift is not None:
        ax = nru.plot_drift(drift, PIXEL_NM)
        ax.set_title(name)
        plt.show()

---
## 4 — Choose the regions

The whole field of view of each slide (Gaussian render of the linked events, camera-pixel grid).
Read off the field of view (panel a) and the ruler to zoom into (panel b) and set them in `VIEWS`.

In [ ]:
OVERVIEW_CLIM = {"80Y": (0, 99.5), "80R": (0, 99.5)}   # colour scale: (low, high) percentiles

for name in RULERS:
    fig, ax, lims = nru.plot_overview(linked[name], acq[name]["width"], acq[name]["height"],
                                      oversampling=2, grid_px=50, clim=OVERVIEW_CLIM[name])
    ax.set_title(f"{name}: linked events", fontsize=9)
    print(f"{name} overview: percentiles {OVERVIEW_CLIM[name]} -> vmin={lims[0]:.4g}, vmax={lims[1]:.4g}")
    plt.show()

---
## 5 — Panels a) and b) of Figures S76 (80Y) and S77 (80R)

a) field of view and b) one ruler, Gaussian renders of the linked localisations (each event
blurred by its own precision). `fov`/`zoom`: `x0`, `y0` (column, row of the lower-left corner)
and `size`, in camera pixels. `SHOW_GRID = True` labels panel a in camera pixels while choosing
the zoom-in; set it to `False` for the figure.

In [ ]:
VIEWS = {
    "80Y": dict(fov=dict(x0=350, y0=300, size=50), zoom=dict(x0=350, y0=304, size=7)),
    "80R": dict(fov=dict(x0=150, y0=200, size=50), zoom=dict(x0=150, y0=202, size=7)),
}
# Colour scale of each panel: (low, high) percentiles of that panel's own render.
CLIM = {
    "80Y": dict(fov=(1, 99.99), zoom=(1, 99.999)),
    "80R": dict(fov=(1, 99.99), zoom=(1, 99.999)),
}
OVERSAMPLING_FOV, OVERSAMPLING_ZOOM = 30, 69   # SR pixels of 2.3 nm and 1 nm
SHOW_GRID = False

for name in RULERS:
    fov, zoom = VIEWS[name]["fov"], VIEWS[name]["zoom"]
    fov_img = nru.render_region(linked[name], **fov, oversampling=OVERSAMPLING_FOV,
                                pixel_nm=PIXEL_NM)
    zoom_img = nru.render_region(linked[name], **zoom, oversampling=OVERSAMPLING_ZOOM,
                                 pixel_nm=PIXEL_NM)
    fig, axs, lims = nru.plot_views(fov_img, zoom_img, fov, zoom, OVERSAMPLING_FOV,
                                    OVERSAMPLING_ZOOM, PIXEL_NM, clim_fov=CLIM[name]["fov"],
                                    clim_zoom=CLIM[name]["zoom"],
                                    grid_px=10 if SHOW_GRID else None)
    print(f"{name}: " + ", ".join(f"{k} percentiles {CLIM[name][k]} -> vmin={v[0]:.4g}, vmax={v[1]:.4g}"
                                  for k, v in lims.items()))
    if not SHOW_GRID:
        plt.savefig(FIG_DIR / f"Nanoruler_{name}_render.svg", dpi=600, format="svg")
    plt.show()

---
## 6 — Panels c) to f): ruler distances

1. **Sites.** DBSCAN of the linked events (`eps_nm`, `min_events` within it; 80R needs a higher
   threshold, as its sites have about twice as many events). A site's centre is the mean of its
   events, the maximum-likelihood centre of a Gaussian blob, known to ~0.3 nm.
2. **Rulers.** Sites with too few events, on for more than 20% of frames, or broader than 40 nm
   are dropped. The rest are grouped at 120 nm, and only groups of exactly three sites whose
   angle at the middle site is at least `MIN_ANGLE_DEG` (in a line) are kept.
3. **Distances.** Centre to centre: neighbouring sites (design 80 nm) and the end sites (160 nm).

c), d) histograms with the design value, the ± `TOLERANCE_NM` band written on the slides and the
median; e) the fraction of distances within a given error of design; f) bias (median − design,
with a 95% bootstrap CI over rulers), SD, robust SD (1.4826 × MAD), RMSE about design, the
fraction within tolerance, and the SD expected from the site-centre uncertainties alone. The
observed spread is several times that expected value, so it is ruler-to-ruler (structural)
variation rather than localisation error.

In [ ]:
SITES = {   # DBSCAN of the linked events into docking sites
    "80Y": dict(eps_nm=10.0, min_events=50),
    "80R": dict(eps_nm=10.0, min_events=200),
}
MIN_ANGLE_DEG = 170.0   # three sites "in a line": angle at the middle site

stats = {}
for name in RULERS:
    sites = nru.find_sites(linked[name], PIXEL_NM, acq[name]["n_frames"], **SITES[name])
    rulers, summary = nru.find_rulers(sites)
    line = rulers[rulers.angle >= MIN_ANGLE_DEG]
    print(f"{name}: {summary['sites']:,} sites ({summary['sites_kept']:,} kept); "
          f"group sizes {summary['group_sizes']}; {len(line):,} of {len(rulers):,} three-site "
          f"groups in a line")
    stats[name] = nru.distance_stats(line, DESIGN_NM, TOLERANCE_NM)
    print(stats[name].round(2).to_string(), "\n")
    info = [("rulers", f"{len(line):,} of {len(rulers):,} three-site groups "
                       f"(angle ≥ {MIN_ANGLE_DEG:g}°)"),
            ("localisation precision",
             f"{linked[name][['xc_err', 'yc_err']].median().mean() * PIXEL_NM:.1f} nm (median)"),
            ("site-centre precision", f"{summary['site_centre_err_nm']:.2f} nm (median)")]
    fig, axs = nru.plot_distances(line, stats[name], DESIGN_NM, TOLERANCE_NM, info=info)
    plt.savefig(FIG_DIR / f"Nanoruler_{name}_distances.svg", dpi=600, format="svg")
    plt.show()
    rulers.to_csv(OUT_ROOT / f"{name}_rulers.csv", index=False)
    stats[name].to_csv(OUT_ROOT / f"{name}_distance_stats.csv")